# Notebook B v12.2 — Nemotron SFT with Assistant-Only Loss — target-module regex fix

Patch after the Kaggle run reached model load but failed at PEFT injection with:

`ValueError: Target modules .*\\.(in_proj|out_proj|up_proj|down_proj)$ not found in the base model.`

Fix: use the correct regex target string in Python:

```python
target_modules = r'.*\.(in_proj|out_proj|up_proj|down_proj)$'
```

The prior notebook used `r'.*\\.(...)$'`, which matched a literal backslash before the dot and therefore matched no modules. This version also prints matched module names before applying PEFT.

In [1]:
import os, re, gc, json, math, time, random, shutil, zipfile, stat, site, sys
from pathlib import Path
from collections import Counter, defaultdict

TRACE_JSONL_NAME = "train_traces_v2_bit.jsonl"
OUTPUT_TAG = "adapter_sft_v2_bit_bal128_asstloss"

SAMPLE_PLAN = {
    "bit_manipulation": 65,
    "gravity": 21,
    "unit_conversion": 21,
    "numeral": 21,
}

RANDOM_SEED = 42
MAX_SEQ_LEN = 384
LR = 2e-4
NUM_EPOCHS = 1
GRAD_ACCUM_STEPS = 1
LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
USE_ASSISTANT_ONLY_LOSS = True
TARGET_MODULES_REGEX = r'.*\.(in_proj|out_proj|up_proj|down_proj)$'

WORKING_DIR = Path('/kaggle/working')
OUTPUT_DIR = WORKING_DIR / OUTPUT_TAG
SMOKE_ZIP_PATH = WORKING_DIR / f"{OUTPUT_TAG}.zip"

print('OUTPUT_TAG:', OUTPUT_TAG)
print('TRACE_JSONL_NAME:', TRACE_JSONL_NAME)
print('SAMPLE_PLAN:', SAMPLE_PLAN)
print('MAX_SEQ_LEN:', MAX_SEQ_LEN)
print('USE_ASSISTANT_ONLY_LOSS:', USE_ASSISTANT_ONLY_LOSS)
print('TARGET_MODULES_REGEX:', TARGET_MODULES_REGEX)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('SMOKE_ZIP_PATH:', SMOKE_ZIP_PATH)

OUTPUT_TAG: adapter_sft_v2_bit_bal128_asstloss
TRACE_JSONL_NAME: train_traces_v2_bit.jsonl
SAMPLE_PLAN: {'bit_manipulation': 65, 'gravity': 21, 'unit_conversion': 21, 'numeral': 21}
MAX_SEQ_LEN: 384
USE_ASSISTANT_ONLY_LOSS: True
TARGET_MODULES_REGEX: .*\.(in_proj|out_proj|up_proj|down_proj)$
OUTPUT_DIR: /kaggle/working/adapter_sft_v2_bit_bal128_asstloss
SMOKE_ZIP_PATH: /kaggle/working/adapter_sft_v2_bit_bal128_asstloss.zip


In [2]:
# -----------------------------
# Kaggle / Nemotron runtime setup — robust CUTLASS path fix
# -----------------------------
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

UTILITY_ROOTS = [
    Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script'),
    Path('/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script'),
    Path('/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script'),
    Path('/tmp'),
]

print('Utility root discovery:')
for root in UTILITY_ROOTS:
    print(' -', root, 'exists=', root.exists())
    if root.exists():
        sys.path.insert(0, str(root))
        site.addsitedir(str(root))
        cutlass_path = root / 'nvidia_cutlass_dsl' / 'python_packages'
        print('   cutlass_path:', cutlass_path, 'exists=', cutlass_path.exists())
        if cutlass_path.exists():
            sys.path.insert(0, str(cutlass_path))
            site.addsitedir(str(cutlass_path))

try:
    import cutlass
    print('CUTLASS import PASS:', getattr(cutlass, '__file__', 'unknown'))
except Exception as e:
    print('CUTLASS import FAILED after path setup:', repr(e))
    print('First 20 sys.path entries:')
    for p in sys.path[:20]:
        print('  ', p)
    raise

for root in UTILITY_ROOTS:
    for src in [root / 'triton/backends/nvidia/bin/ptxas', root / 'triton/backends/nvidia/bin/ptxas-blackwell']:
        if src.exists():
            dst = Path('/tmp') / src.name
            shutil.copy2(src, dst)
            os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
            print('Prepared executable:', dst)
            if dst.name == 'ptxas':
                os.environ['TRITON_PTXAS_PATH'] = str(dst)
            if dst.name == 'ptxas-blackwell':
                os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = str(dst)

try:
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Patched Triton ptxas version check.')
except Exception as e:
    print('Triton patch skipped:', repr(e))

def patch_nemotron_fast_path():
    patched = []
    for name, mod in list(sys.modules.items()):
        if 'modeling_nemotron_h' in name and hasattr(mod, 'is_fast_path_available'):
            try:
                mod.is_fast_path_available = False
                patched.append(name)
            except Exception:
                pass
    if patched:
        print('Patched fast path modules:', patched)

patch_nemotron_fast_path()

Utility root discovery:
 - /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script exists= True
   cutlass_path: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages exists= True
 - /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script exists= False
 - /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script exists= False
 - /tmp exists= True
   cutlass_path: /tmp/nvidia_cutlass_dsl/python_packages exists= False
CUTLASS import PASS: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/cutlass/__init__.py
Prepared executable: /tmp/ptxas
Prepared executable: /tmp/ptxas-blackwell
Patched Triton ptxas version check.


In [3]:
def find_trace_jsonl(name: str) -> Path:
    roots = [Path('/kaggle/input/notebooks'), Path('/kaggle/input'), WORKING_DIR]
    hits = []
    for root in roots:
        if root.exists():
            hits.extend(root.glob(f'**/{name}'))
    hits = sorted(set(hits), key=lambda p: str(p))
    print('Trace candidates:')
    for h in hits[:20]:
        print(' -', h)
    if not hits:
        raise FileNotFoundError(f'Could not find {name}')
    return hits[0]

TRACE_PATH = find_trace_jsonl(TRACE_JSONL_NAME)
print('Using TRACE_PATH:', TRACE_PATH)
records = []
with open(TRACE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print('Loaded trace records:', len(records))
print('Category counts:', Counter(r.get('category', 'UNKNOWN') for r in records))
print('Example keys:', sorted(records[0].keys()))
print('Example category:', records[0].get('category'))

Trace candidates:
 - /kaggle/input/notebooks/jatalepawan/nemotron-trace-generator-v2-bit-20260506/train_traces_v2_bit.jsonl
Using TRACE_PATH: /kaggle/input/notebooks/jatalepawan/nemotron-trace-generator-v2-bit-20260506/train_traces_v2_bit.jsonl
Loaded trace records: 5635
Category counts: Counter({'gravity': 1597, 'unit_conversion': 1594, 'numeral': 1576, 'bit_manipulation': 868})
Example keys: ['answer', 'approx_trace_chars', 'category', 'id', 'messages', 'trace_version']
Example category: bit_manipulation


In [4]:
random.seed(RANDOM_SEED)
by_cat = defaultdict(list)
for r in records:
    by_cat[r.get('category', 'UNKNOWN')].append(r)

selected = []
for cat, n in SAMPLE_PLAN.items():
    pool = list(by_cat.get(cat, []))
    if len(pool) < n:
        raise ValueError(f'Not enough records for {cat}: requested {n}, available {len(pool)}')
    rng = random.Random(RANDOM_SEED + abs(hash(cat)) % 100000)
    rng.shuffle(pool)
    selected.extend(pool[:n])
random.Random(RANDOM_SEED).shuffle(selected)
print('Selected records:', len(selected))
print('Selected category counts:', Counter(r.get('category', 'UNKNOWN') for r in selected))
assert len(selected) == sum(SAMPLE_PLAN.values())
print('Sampling validation passed.')

Selected records: 128
Selected category counts: Counter({'bit_manipulation': 65, 'unit_conversion': 21, 'gravity': 21, 'numeral': 21})
Sampling validation passed.


In [5]:
import torch
import kagglehub
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
print('MODEL_PATH:', MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, trust_remote_code=True, dtype=torch.bfloat16, device_map={'': 0}
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16, device_map={'': 0}
    )

patch_nemotron_fast_path()
model.config.use_cache = False

# Validate target-module regex before PEFT injection.
target_re = re.compile(TARGET_MODULES_REGEX)
matched_names = [name for name, _ in model.named_modules() if target_re.match(name)]
print('Matched LoRA target modules:', len(matched_names))
print('First 20 matched targets:')
for name in matched_names[:20]:
    print(' -', name)
if not matched_names:
    print('Sample module names for debugging:')
    for name, _ in list(model.named_modules())[:200]:
        print(' -', name)
    raise ValueError(f'No modules matched TARGET_MODULES_REGEX={TARGET_MODULES_REGEX}')

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES_REGEX,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Model + LoRA ready.')

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Patched fast path modules: ['transformers_modules._1.modeling_nemotron_h']
Matched LoRA target modules: 5980
First 20 matched targets:
 - backbone.layers.0.mixer.in_proj
 - backbone.layers.0.mixer.out_proj
 - backbone.layers.1.mixer.experts.0.up_proj
 - backbone.layers.1.mixer.experts.0.down_proj
 - backbone.layers.1.mixer.experts.1.up_proj
 - backbone.layers.1.mixer.experts.1.down_proj
 - backbone.layers.1.mixer.experts.2.up_proj
 - backbone.layers.1.mixer.experts.2.down_proj
 - backbone.layers.1.mixer.experts.3.up_proj
 - backbone.layers.1.mixer.experts.3.down_proj
 - backbone.layers.1.mixer.experts.4.up_proj
 - backbone.layers.1.mixer.experts.4.down_proj
 - backbone.layers.1.mixer.experts.5.up_proj
 - backbone.layers.1.mixer.experts.5.down_proj
 - backbone.layers.1.mixer.experts.6.up_proj
 - backbone.layers.1.mixer.experts.6.down_proj
 - backbone.layers.1.mixer.experts.7.up_proj
 - backbone.layers.1.mixer.experts.7.down_proj
 - backbone.layers.1.mixer.experts.8.up_proj
 - backbone.l

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116
Model + LoRA ready.


In [6]:
def boxed(answer):
    s = str(answer).strip()
    return s if '\\boxed{' in s else f'\\boxed{{{s}}}'

def record_to_messages(rec):
    if isinstance(rec.get('messages'), list) and len(rec['messages']) >= 2:
        return rec['messages']
    prompt = rec.get('prompt') or rec.get('user') or rec.get('question') or rec.get('input')
    assistant = rec.get('assistant') or rec.get('response') or rec.get('completion') or rec.get('solution')
    answer = rec.get('answer')
    if assistant is None:
        assistant = boxed(answer)
    elif answer is not None and '\\boxed{' not in str(assistant):
        assistant = str(assistant).rstrip() + '\n\nFinal answer: ' + boxed(answer)
    if prompt is None:
        return None
    user_msg = str(prompt).rstrip() + '\nPlease put your final answer inside `\\boxed{}`.'
    return [{'role': 'user', 'content': user_msg}, {'role': 'assistant', 'content': str(assistant).strip()}]

def encode_record(rec):
    messages = record_to_messages(rec)
    if messages is None:
        text = rec.get('text') or rec.get('formatted_text')
        if not text:
            raise ValueError(f'Cannot build text for record keys={sorted(rec.keys())}')
        enc = tokenizer(str(text), truncation=True, max_length=MAX_SEQ_LEN, return_tensors='pt')
        labels = enc['input_ids'].clone()
        return enc['input_ids'][0], enc['attention_mask'][0], labels[0], 'full_text_fallback'

    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    full = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LEN, return_tensors='pt')
    input_ids = full['input_ids'][0]
    attention_mask = full['attention_mask'][0]
    labels = input_ids.clone()
    if USE_ASSISTANT_ONLY_LOSS:
        user_only = [m for m in messages if m.get('role') != 'assistant']
        prefix_text = tokenizer.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
        prefix_ids = tokenizer(prefix_text, add_special_tokens=False, return_tensors='pt')['input_ids'][0]
        prefix_len = min(len(prefix_ids), len(labels))
        labels[:prefix_len] = -100
        if (labels != -100).sum().item() == 0:
            return None
    return input_ids, attention_mask, labels, 'assistant_only' if USE_ASSISTANT_ONLY_LOSS else 'full_text'

encoded = []
skipped = 0
modes = Counter()
for rec in selected:
    item = encode_record(rec)
    if item is None:
        skipped += 1
        continue
    encoded.append((rec, *item))
    modes[item[-1]] += 1
print('Encoded examples:', len(encoded), 'skipped:', skipped)
print('Label modes:', modes)
assert encoded
rec, input_ids, attention_mask, labels, mode = encoded[0]
print('First encoded length:', len(input_ids), 'mode:', mode, 'category:', rec.get('category'))
print('Train-label tokens:', int((labels != -100).sum().item()))

Encoded examples: 128 skipped: 0
Label modes: Counter({'assistant_only': 128})
First encoded length: 384 mode: assistant_only category: bit_manipulation
Train-label tokens: 125


In [7]:
model.train()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
losses = []
step = 0
start = time.time()
for epoch in range(NUM_EPOCHS):
    random.Random(RANDOM_SEED + epoch).shuffle(encoded)
    for i, (rec, input_ids, attention_mask, labels, mode) in enumerate(encoded, start=1):
        input_ids = input_ids.unsqueeze(0).to(model.device)
        attention_mask = attention_mask.unsqueeze(0).to(model.device)
        labels = labels.unsqueeze(0).to(model.device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()
        if i % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1
        losses.append(float(loss.detach().cpu()) * GRAD_ACCUM_STEPS)
        if i == 1 or i % 10 == 0 or i == len(encoded):
            elapsed = time.time() - start
            print(f'epoch={epoch+1} item={i}/{len(encoded)} step={step} loss={losses[-1]:.4f} elapsed={elapsed:.1f}s cat={rec.get("category")}')
        del outputs, loss, input_ids, attention_mask, labels
        if i % 25 == 0:
            gc.collect()
            torch.cuda.empty_cache()
print('Training complete.')
print('Mean loss:', sum(losses) / len(losses))
print('Last 10 mean loss:', sum(losses[-10:]) / min(10, len(losses)))

epoch=1 item=1/128 step=1 loss=1.4997 elapsed=8.7s cat=gravity
epoch=1 item=10/128 step=10 loss=1.4430 elapsed=30.3s cat=numeral
epoch=1 item=20/128 step=20 loss=0.0950 elapsed=54.2s cat=unit_conversion
epoch=1 item=30/128 step=30 loss=0.2860 elapsed=78.1s cat=bit_manipulation
epoch=1 item=40/128 step=40 loss=0.1514 elapsed=102.3s cat=gravity
epoch=1 item=50/128 step=50 loss=0.0914 elapsed=125.8s cat=gravity
epoch=1 item=60/128 step=60 loss=0.0924 elapsed=150.4s cat=unit_conversion
epoch=1 item=70/128 step=70 loss=0.3899 elapsed=173.7s cat=gravity
epoch=1 item=80/128 step=80 loss=0.0940 elapsed=198.9s cat=gravity
epoch=1 item=90/128 step=90 loss=0.1266 elapsed=222.7s cat=bit_manipulation
epoch=1 item=100/128 step=100 loss=0.0001 elapsed=246.2s cat=numeral
epoch=1 item=110/128 step=110 loss=0.2081 elapsed=269.6s cat=numeral
epoch=1 item=120/128 step=120 loss=0.0004 elapsed=293.5s cat=numeral
epoch=1 item=128/128 step=128 loss=0.1357 elapsed=313.4s cat=unit_conversion
Training complete.


In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tok_dbg = OUTPUT_DIR / 'tokenizer_debug'
tok_dbg.mkdir(exist_ok=True)
try:
    tokenizer.save_pretrained(tok_dbg)
except Exception as e:
    print('Tokenizer debug save skipped:', repr(e))
print('Saved adapter files:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print(' -', p.relative_to(OUTPUT_DIR), f'{p.stat().st_size/1024/1024:.2f} MB')
if SMOKE_ZIP_PATH.exists():
    SMOKE_ZIP_PATH.unlink()
with zipfile.ZipFile(SMOKE_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['adapter_config.json', 'adapter_model.safetensors', 'adapter_model.bin']:
        p = OUTPUT_DIR / fname
        if p.exists():
            zf.write(p, arcname=fname)
with zipfile.ZipFile(SMOKE_ZIP_PATH, 'r') as zf:
    names = zf.namelist()
print('Created adapter debug zip:', SMOKE_ZIP_PATH, f'{SMOKE_ZIP_PATH.stat().st_size/1024/1024:.2f} MB')
print('Zip contents:', names)
assert 'adapter_config.json' in names
assert ('adapter_model.safetensors' in names) or ('adapter_model.bin' in names)
print('Notebook B complete. Use this exact tag in Notebook C:', OUTPUT_TAG)

Saved adapter files:
 - README.md 0.01 MB
 - adapter_config.json 0.00 MB
 - adapter_model.safetensors 3359.18 MB
 - tokenizer_debug/chat_template.jinja 0.01 MB
 - tokenizer_debug/tokenizer.json 16.29 MB
 - tokenizer_debug/tokenizer_config.json 0.00 MB
Created adapter debug zip: /kaggle/working/adapter_sft_v2_bit_bal128_asstloss.zip 2988.55 MB
Zip contents: ['adapter_config.json', 'adapter_model.safetensors']
Notebook B complete. Use this exact tag in Notebook C: adapter_sft_v2_bit_bal128_asstloss
